# Features

Machine learning is not just about the models themselves -- so much of machine learning performance and ability **is about data**.

In ML parlance we have our *labels* (what we're trying to predict) and *features* (the data that we're using to predict it). To make this concrete let's start by working with a structured dataset (i.e. an excel datasheet). 

To develop this understanding we can work with a simple structured dataset about student academic performance in college (`GPA`) given their high school achievements and some other simple information. 

In [ ]:
import pandas as pd

df =  pd.read_csv('../../data/structured_data/student_gpa_data.csv')
df.head()

Now we don't have to use all of the data columns. We could just train a simple model with High School GPA and SAT scores. 

In [ ]:
len(X), len(y)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

#Create our X and Y
n_lim = int(len(df.HSGPA)*.7)

X = df.loc[:n_lim, ['HSGPA', 'SATV', 'SATM']]
y = df.loc[:n_lim, ['GPA']]['GPA']

rfr = RandomForestRegressor()
#Create the parameter grid
param_grid = {'n_estimators': [50, 100, 500],
              'max_depth': [5, 10, 20],
              'min_samples_split': [2, 4, 8]}
sh = HalvingGridSearchCV(rfr, param_grid, factor=3).fit(X, y)
sh.best_params_

And now we can actually evaluate the performance of the best model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rfr = RandomForestRegressor(n_estimators=100, max_depth=5, min_samples_split=8)
rfr.fit(X, y)

#Now make the predictions
tX = df.loc[n_lim:, ['HSGPA', 'SATV', 'SATM']]
ty = df.loc[n_lim:, ['GPA']]['GPA']

preds = rfr.predict(tX)

#Metrics
print('MSE ', mean_squared_error(ty, preds))
print('MAE ', mean_absolute_error(ty, preds))
print('R2 ', r2_score(ty, preds))

So generally pretty poor performance as it is. But we can throw more data at it to try and improve the performance (i.e. increasing our features)

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

#Create our X and Y
n_lim = int(len(df.HSGPA)*.7)

X = df.loc[:n_lim, [x for x in df.columns if x != 'GPA']]
y = df.loc[:n_lim, ['GPA']]['GPA']

rfr = RandomForestRegressor()
#Create the parameter grid
param_grid = {'n_estimators': [50, 100, 500],
              'max_depth': [5, 10, 20],
              'min_samples_split': [4, 8, 20]}
sh = HalvingGridSearchCV(rfr, param_grid, factor=3).fit(X, y)
sh.best_params_

And test the entire model

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rfr = RandomForestRegressor(n_estimators=100, min_samples_split=4, max_depth=5)
rfr.fit(X, y)

#Now make the predictions
tX = df.loc[n_lim:, [x for x in df.columns if x != 'GPA']]
ty = df.loc[n_lim:, ['GPA']]['GPA']

preds = rfr.predict(tX)

#Metrics
print('MSE ', mean_squared_error(ty, preds))
print('MAE ', mean_absolute_error(ty, preds))
print('R2 ', r2_score(ty, preds))

So the increased data did not help our model much. But we can of course generate additional features given the existing data.

In [ ]:
df.head()

So we could try to emphasize the entire SAT score to make this prediction 

In [ ]:
df['SAT'] = df['SATV'] + df['SATM']
df.head()

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

#Create our X and Y
n_lim = int(len(df.HSGPA)*.7)

X = df.loc[:n_lim, [x for x in df.columns if x != 'GPA']]
y = df.loc[:n_lim, ['GPA']]['GPA']

rfr = RandomForestRegressor()
#Create the parameter grid
param_grid = {'n_estimators': [25, 50, 100, 250, 500],
              'max_depth': [2, 3, 5, 10, 20],
              'min_samples_split': [2, 4, 8, 14, 20]}
sh = HalvingGridSearchCV(rfr, param_grid, factor=3).fit(X, y)
sh.best_params_

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

rfr = RandomForestRegressor(n_estimators=50, min_samples_split=8, max_depth=3)
rfr.fit(X, y)

#Now make the predictions
tX = df.loc[n_lim:, [x for x in df.columns if x != 'GPA']]
ty = df.loc[n_lim:, ['GPA']]['GPA']

preds = rfr.predict(tX)

#Metrics
print('MSE ', mean_squared_error(ty, preds))
print('MAE ', mean_absolute_error(ty, preds))
print('R2 ', r2_score(ty, preds))

And now we've made the model worse becase of our additional feature. More features aren't always better! 

# Back to text

Our original predictions of sentiment were not great...

And a lot of that has to do with the features that we were using. We constructed our `X` dataset for prediction with every single word in the vocabulary across the entire corpus. 

Most of those words don't carry any meaningful information in trying to make a prediction of whether the paragraph has a negative, positive, or neutral sentiment! Text requires us to perform feature 'extraction', i.e. generating only meaningful features that we use to try and use in a prediction problem. 

One of the most common methods to perform this feature extraction/generation is TF-IDF

# TF-IDF

One of the most used processing steps when transforming unstructured text into a prediction problem is TF-IDF (Term Frequency-Inverse Document Frequency). This process involves the switch from counts to frequencies and the comparison of the term overall frequency of the term to its frequency within a document. There's more than operationalization of these terms, but for ease of use we'll use:

$tf(t,d) = \frac{f(t,d)}{\sum_{t\in d} 1}$

$idf(t, D) = log\frac{N}{|{d\in D:t\in d}|}$

where $t$ is a term, $d$ is a single document, $N$  is the total number of documents in the corpus, and $|{d\in D:t\in d}|$ is the number of documents where term $t$ appears. The final calculation is thus:

$tf * idf$

What does this do for us?

In [1]:
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
import pandas as pd

traindf = pd.read_csv('../../data/text_data/PerSenT/train.csv')
traindf.head()

stop = set(stopwords.words('english'))

vectorizer = TfidfVectorizer (max_features=2500, stop_words= list(stop) )

#Pull the texts outdd
texts = []
for doc in traindf.DOCUMENT:
    texts.append( doc )

features = vectorizer.fit_transform(texts).toarray()

ModuleNotFoundError: No module named 'nltk'

And since we already know that we need to have a test set, let's do that

In [ ]:
from sklearn.model_selection import train_test_split

y = le.fit_transform(traindf.TRUE_SENTIMENT)

Xtrain, Xtest, ytrain, ytest = train_test_split(features, y, test_size=0.2, random_state=9)

And re-run our model

In [ ]:
from sklearn.metrics import classification_report, accuracy_score

clf = RandomForestClassifier(n_estimators=100)
clf.fit(Xtrain, ytrain)
ypred = clf.predict(Xtest)

print(classification_report(ytest, ypred, target_names = ['Negative', 'Neutral', 'Positive']))
print(accuracy_score(ytest, ypred))

Hey! We got better! of course, we can try to improve performance through the optimization of the model parameters. 

In [ ]:
#For time's sake in class, we'll use a halving grid search
from sklearn.experimental import enable_halving_search_cv
from sklearn.model_selection import HalvingGridSearchCV

#Start up the model
base_estimator = RandomForestClassifier()
#Create the parameter grid
param_grid = {'n_estimators': [50, 100, 500],
              'max_depth': [5, 10, 20],
              'min_samples_split': [2, 4, 8]}
sh = HalvingGridSearchCV(base_estimator, param_grid, cv=3, factor=3).fit(Xtrain, ytrain)

In [ ]:
sh.n_candidates_

In [ ]:
sh.best_params_

In [ ]:
sh.best_score_